# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda

Vamos trabalhar com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [13]:
import numpy as np
import pandas as pd
import patsy
import statsmodels.api as sm

In [45]:
df = pd.read_csv('previsao_de_renda.csv')

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
dtypes:

1. Ajuste um modelo para prever log(renda) considerando todas as covariáveis disponíveis.
    - Utilizando os recursos do Patsy, coloque as variáveis qualitativas como *dummies*.
    - Mantenha sempre a categoria mais frequente como casela de referência
    - Avalie os parâmetros e veja se parecem fazer sentido prático.

2. Remova a variável menos significante e analise:
    - Observe os indicadores que vimos, e avalie se o modelo melhorou ou piorou na sua opinião.
    - Observe os parâmetros e veja se algum se alterou muito.

3. Siga removendo as variáveis menos significantes, sempre que o *p-value* for menor que 5%. Compare o modelo final com o inicial. Observe os indicadores e conclua se o modelo parece melhor. 
    

In [49]:
df = df.drop(columns=['Unnamed: 0', 'data_ref', 'id_cliente'])
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   sexo                   15000 non-null  object 
 1   posse_de_veiculo       15000 non-null  bool   
 2   posse_de_imovel        15000 non-null  bool   
 3   qtd_filhos             15000 non-null  int64  
 4   tipo_renda             15000 non-null  object 
 5   educacao               15000 non-null  object 
 6   estado_civil           15000 non-null  object 
 7   tipo_residencia        15000 non-null  object 
 8   idade                  15000 non-null  int64  
 9   tempo_emprego          12427 non-null  float64
 10  qt_pessoas_residencia  15000 non-null  float64
 11  renda                  15000 non-null  float64
dtypes: bool(2), float64(3), int64(2), object(5)
memory usage: 1.2+ MB
None


# Exercício 1

In [51]:
# Calculando o log(renda)
df['log_renda'] = np.log(df['renda'])

# Criando a fórmula usando Patsy para incluir dummies (com categorias de referência)
formula = "log_renda ~ " + " + ".join([
    f"C({col}, Treatment)" if df[col].dtype == 'object' else col
    for col in df.columns if col not in ['Unnamed: 0', 'data_ref', 'id_cliente', 'renda', 'log_renda']
])

# Criando os dados de design (X e y)
y, X = patsy.dmatrices(formula, data=df, return_type='dataframe')

# Ajustando o modelo de regressão
model = sm.OLS(y, X).fit()

# Exibindo o resumo dos parâmetros do modelo
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.357
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     287.5
Date:                Wed, 19 Mar 2025   Prob (F-statistic):               0.00
Time:                        09:46:22   Log-Likelihood:                -13568.
No. Observations:               12427   AIC:                         2.719e+04
Df Residuals:                   12402   BIC:                         2.737e+04
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

Os grupos utilizados como referência do modelo parece ter demonstrado uma diferença significativa para uma quantidade considerável de variáveis. O R² quadrado também está razoável, considerando que estamos falando de pessoas para prever a renda no modelo.

# Exercício 2

In [57]:
# Atualizando manualmente a fórmula para remover 'educacao'
formula_atualizada = "log_renda ~ " + " + ".join([
    col for col in df.columns if col not in ['Unnamed: 0', 'data_ref', 'id_cliente', 'renda', 'log_renda', 'educacao']
])

# Criando os dados de design novamente
y, X = patsy.dmatrices(formula_atualizada, data=df, return_type='dataframe')

# Ajustando o modelo atualizado
model_atualizado = sm.OLS(y, X).fit()

# Comparando indicadores e imprimindo o resumo do modelo
print(model_atualizado.summary())

                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.354
Model:                            OLS   Adj. R-squared:                  0.353
Method:                 Least Squares   F-statistic:                     340.0
Date:                Wed, 19 Mar 2025   Prob (F-statistic):               0.00
Time:                        09:47:51   Log-Likelihood:                -13601.
No. Observations:               12427   AIC:                         2.724e+04
Df Residuals:                   12406   BIC:                         2.740e+04
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

In [59]:
# Atualizando manualmente a fórmula para remover 'educacao'
formula_atualizada = "log_renda ~ " + " + ".join([
    col for col in df.columns if col not in ['Unnamed: 0', 'data_ref', 'id_cliente', 'renda', 'log_renda', 'educacao', 'tipo_residencia']
])

# Criando os dados de design novamente
y, X = patsy.dmatrices(formula_atualizada, data=df, return_type='dataframe')

# Ajustando o modelo atualizado
model_atualizado = sm.OLS(y, X).fit()

# Comparando indicadores e imprimindo o resumo do modelo
print(model_atualizado.summary())

                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.354
Model:                            OLS   Adj. R-squared:                  0.353
Method:                 Least Squares   F-statistic:                     453.1
Date:                Wed, 19 Mar 2025   Prob (F-statistic):               0.00
Time:                        09:49:31   Log-Likelihood:                -13603.
No. Observations:               12427   AIC:                         2.724e+04
Df Residuals:                   12411   BIC:                         2.736e+04
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

# Exercício 3

Eu segui removendo todas as que não tivessem diferença significativa entre grupos, em algumas variáveis há diferença significativa entre alguns grupos e outros não quando comparado ao grupo referência. Contudo, reduzir as variáveis houve uma piora do AIC e do R², embora não tenha sido uma grande diferença, talvez nesse contexto valeria utilizar o modelo mais simples.